# Pedestrian Detection with Spiking Neural Networks (SNNs)

### **Notebook goal**

The goal of this notebook is the implementation and evaluation of a **Spiking Neural Network (SNN)** model applied to a **Pedestrian Detection** task. The workflow is structured to reflect the core practices of Neuromorphic Machine Learning Operations (NMLOps).

## 1. Import and Basic Settings

In this section, all the necessary libraries to execute the experiment are loaded: file system management, XML annotation parsing, image manipulation (PIL), Deep Learning modules (PyTorch), and neuromorphic computing components (`snntorch`).

In [1]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

## 2. Global Configurations & Hyperparameters

Centralized definition of global settings, dataset paths, and training hyperparameters for the SNN architecture, ensuring experiment reproducibility.

In [2]:
# Hardware device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# System paths for dataset components
TRAIN_DIR = "data/train" 
FRAMES_DIR = os.path.join(TRAIN_DIR, "Pedestrian frame")
LABELS_DIR = os.path.join(TRAIN_DIR, "Pedestrian label")

# Definition of main SNN and optimization hyperparameters
num_steps = 10      # Time steps for SNN simulation
batch_size = 16     # Batch size
num_epochs = 20     # Training epochs
learning_rate = 1e-3

## 3. Neuromorphic Dataset Management

Implementation of the custom `PedestrianDataset` module. This class handles the lazy loading of frames and the parsing of XML files containing presence tags and bounding box vector coordinates for pedestrians.

In [3]:
class PedestrianDataset(Dataset):
    def __init__(self, frames_dir, labels_dir, transform=None):
        self.frames_dir = frames_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.filenames = [os.path.splitext(f)[0] for f in os.listdir(frames_dir) if f.endswith(('.jpg', '.png'))]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        img_path = os.path.join(self.frames_dir, name + '.jpg')
        if not os.path.exists(img_path):
            img_path = os.path.join(self.frames_dir, name + '.png')
        
        xml_path = os.path.join(self.labels_dir, name + '.xml')
        
        image = Image.open(img_path).convert("RGB")
        
        # Base default in case of pedestrian absence
        label_cls = 0.0
        bbox = [0.0, 0.0, 0.0, 0.0]
        
        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            # Check presence of 'object' tag (pedestrian)
            obj = root.find('object')
            if obj is not None:
                label_cls = 1.0
                bndbox = obj.find('bndbox')
                if bndbox is not None:
                    xmin = float(bndbox.find('xmin').text)
                    ymin = float(bndbox.find('ymin').text)
                    xmax = float(bndbox.find('xmax').text)
                    ymax = float(bndbox.find('ymax').text)
                    bbox = [xmin, ymin, xmax, ymax]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label_cls, dtype=torch.float32), torch.tensor(bbox, dtype=torch.float32)

### 3.1 Data Augmentation & Data Loader Setup

Preparation of the transformation pipelines (resizing and tensor normalization) and partitioning of the dataset into the canonical Training, Validation, and Testing fractions.

In [4]:
# Pipeline for graphical transformation of input images
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Initialization of the full dataset
full_dataset = PedestrianDataset(FRAMES_DIR, LABELS_DIR, transform=transform)
total_size = len(full_dataset)

# Deterministic splitting of shares (80% Train, 10% Val, 10% Test)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Generation of iterable DataLoader channels
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## 4. Spiking Neural Network Architecture

Definition of the Spiking Neural Network architecture (`PedestrianSNN`). The model integrates convolutional layers for spatial feature extraction coupled with Leaky Integrate-and-Fire (`snn.Leaky`) artificial neurons, managing classification and bounding box regression in parallel through recurrent time steps.

In [5]:
class PedestrianSNN(nn.Module):
    def __init__(self, num_steps):
        super(PedestrianSNN, self).__init__()
        self.num_steps = num_steps
        
        # Definition of the surrogate gradient to overcome spike non-differentiability
        spike_grad = surrogate.fast_sigmoid()
        beta = 0.9  # Neuron membrane decay constant
        
        # Spatial Feature Extraction (Convolutional Layers)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.pool1 = nn.MaxPool2d(2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.pool2 = nn.MaxPool2d(2)
        
        # Fully Connected layers for processing decision flows
        # Expected output after pools: 32 feature maps of 32x32 size
        self.fc_shared = nn.Linear(32 * 32 * 32, 128)
        self.lif_shared = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        # Classification Head (Pedestrian presence)
        self.fc_cls = nn.Linear(128, 1)
        self.lif_cls = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        # Regression Head (Bounding Box spatial coordinates)
        self.fc_bbox = nn.Linear(128, 4)
        self.lif_bbox = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        # Initial reset of internal membrane potentials for SNN components
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem_shared = self.lif_shared.init_leaky()
        mem_cls = self.lif_cls.init_leaky()
        mem_bbox = self.lif_bbox.init_leaky()
        
        # Accumulators for simulation outputs across time steps
        cls_out_rec = []
        bbox_out_rec = []
        
        for step in range(self.num_steps):
            # Pass through the first Convolutional + Spiking block
            cur1 = self.pool1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            
            # Pass through the second Convolutional + Spiking block
            cur2 = self.pool2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            
            # Flattening for feeding into dense layers
            spk2_flat = spk2.view(spk2.size(0), -1)
            
            # Shared dense layer
            cur_shared = self.fc_shared(spk2_flat)
            spk_shared, mem_shared = self.lif_shared(cur_shared, mem_shared)
            
            # Generation of classification and localization outputs for the current step
            cur_cls = self.fc_cls(spk_shared)
            spk_cls, mem_cls = self.lif_cls(cur_cls, mem_cls)
            cls_out_rec.append(mem_cls)  # Using potential as a continuous value
            
            cur_bbox = self.fc_bbox(spk_shared)
            spk_bbox, mem_bbox = self.lif_bbox(cur_bbox, mem_bbox)
            bbox_out_rec.append(mem_bbox)
            
        # Stack temporal records and average over executed steps
        cls_output = torch.stack(cls_out_rec, dim=0).mean(dim=0).squeeze(-1)
        bbox_output = torch.stack(bbox_out_rec, dim=0).mean(dim=0)
        
        return cls_output, bbox_output

## 5. Model Training and Validation Pipeline

Configuration of loss functions (`BCEWithLogitsLoss` combined with `MSELoss`) and the iterative training and validation loop for updating synaptic weights.

In [6]:
# Model instantiation and allocation to the correct device
model = PedestrianSNN(num_steps=num_steps).to(device)

# Definition of error calculation criteria and the Adam optimizer
criterion_cls = nn.BCEWithLogitsLoss()
criterion_bbox = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Main Training & Validation Loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels_cls, bboxes in train_loader:
        images = images.to(device)
        labels_cls = labels_cls.to(device)
        bboxes = bboxes.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        pred_cls, pred_bbox = model(images)
        
        # Combined loss calculation (Classification + Bounding Box Regression)
        loss_cls = criterion_cls(pred_cls, labels_cls)
        loss_box = criterion_bbox(pred_bbox, bboxes)
        loss = loss_cls + 0.01 * loss_box
        
        # Backpropagation stages and parameter optimization
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")

Epoch [1/20], Training Loss: 96.3253
Epoch [2/20], Training Loss: 41.6663
Epoch [3/20], Training Loss: 30.8639
Epoch [4/20], Training Loss: 26.7819
Epoch [5/20], Training Loss: 24.8772
Epoch [6/20], Training Loss: 23.8281
Epoch [7/20], Training Loss: 23.3977
Epoch [8/20], Training Loss: 22.7785
Epoch [9/20], Training Loss: 22.2984
Epoch [10/20], Training Loss: 21.8481
Epoch [11/20], Training Loss: 21.5159
Epoch [12/20], Training Loss: 21.2490
Epoch [13/20], Training Loss: 21.0237
Epoch [14/20], Training Loss: 20.6185
Epoch [15/20], Training Loss: 20.3063
Epoch [16/20], Training Loss: 20.2388
Epoch [17/20], Training Loss: 19.9841
Epoch [18/20], Training Loss: 19.7770
Epoch [19/20], Training Loss: 19.5281
Epoch [20/20], Training Loss: 19.3140


## 6. Performance Evaluation & Metrics

Definition of analytic metrics (Accuracy and Intersection over Union - IoU) needed to quantify model performance on the independent test split, followed by the execution of the final verification.

In [7]:
def calculate_iou(boxA, boxB):
    # Calculation of rectangle intersection coordinates
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    interArea = max(0, xB - xA) * max(0, yB - yA)
    
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    unionArea = boxAArea + boxBArea - interArea
    if unionArea == 0:
        return 0.0
        
    return interArea / unionArea

def test_model_accuracy(model, test_loader, device):
    model.eval()
    correct_cls = 0
    total_images = 0
    total_iou = 0.0
    
    with torch.no_grad():
        for images, labels_cls, bboxes in test_loader:
            images = images.to(device)
            labels_cls = labels_cls.cpu()
            bboxes = bboxes.cpu()
            
            pred_cls, pred_bbox = model(images)
            pred_cls = pred_cls.cpu()
            pred_bbox = pred_bbox.cpu()
            
            # Conversion of continuous logits to binary decisions (threshold at 0.0 for logits)
            preds_binary = (pred_cls >= 0.0).float()
            correct_cls += (preds_binary == labels_cls).sum().item()
            
            total_images += images.size(0)
            
            for i in range(images.size(0)):
                if labels_cls[i] == 1.0 and preds_binary[i] == 1.0:
                    total_iou += calculate_iou(pred_bbox[i].tolist(), bboxes[i].tolist())
                    
    cls_accuracy = (correct_cls / total_images) * 100
    avg_iou = (total_iou / total_images) * 100
    
    print("\n================ TEST SPLIT RESULTS ================")
    print(f"Total Evaluated Images: {total_images}")
    print(f"Pedestrian Presence Accuracy: {cls_accuracy:.2f}%")
    print(f"Average Bounding Box IoU: {avg_iou:.2f}%")
    print("====================================================")
    
    return cls_accuracy, avg_iou

### 6.1 Inference Evaluation Execution

Final launch of the test on the trained model to gather quantitative results regarding accuracy and spatial overlap.

In [8]:
# Final execution of the neuromorphic architecture test
test_model_accuracy(model, test_loader, device)


================ TEST SPLIT RESULTS ================
Total Evaluated Images: 476
Pedestrian Presence Accuracy: 91.60%
Average Bounding Box IoU: 19.84%


(91.59663865546219, 19.840389138114944)